In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"

processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name).to(device)

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/661M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/230 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
projector.weight           | MISSING    | 
classifier.weight          | MISSING    | 
classifier.bias            | MISSING    | 
projector.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
df_features_for_model = pd.read_feather('df_features_for_model')

In [ ]:
directory = './data/'

In [6]:
class AudioDataSet(Dataset):

    def __init__(self, paths, sr=16000, duration=30):

        self.paths = paths
        self.sr = sr
        self.duration = duration

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        y, sr = librosa.load(self.paths[idx], sr=self.sr, mono=True, duration=self.duration)
        y = np.pad(y, (0, 480000-len(y)))
        y = torch.tensor(y, dtype=torch.float32)

        return {
            'path': self.paths[idx].replace(directory, ''), 
            'audio': y.to(device)
        }

In [7]:
data = AudioDataSet((directory + df_features_for_model['path']).tolist())

In [8]:
loader = DataLoader(data, batch_size=16)

In [9]:
def extract_emotion_features(y, processor, model, sr=16000):

    y_proc = processor(y, sampling_rate=sr)
    y_proc = y_proc['input_values'][0]
    # y_proc = y_proc.reshape(1, -1)
    y_proc = torch.from_numpy(y_proc).to(device)

    with torch.no_grad():
        y_proc = model(y_proc).logits

    y_proc = y_proc.detach().cpu().numpy().squeeze()

    return y_proc

In [10]:
results = dict()
for batch in tqdm(loader):
    values = extract_emotion_features(batch['audio'], processor, model, sr=16000)
    _dct = dict(zip(batch['path'], values))
    results.update(_dct)

100%|██████████| 976/976 [1:35:39<00:00,  5.88s/it]


In [11]:
pd.DataFrame.from_dict(results, orient='index').reset_index().to_feather('df_feature_emotions')

/usr/local/lib/python3.12/dist-packages/pyarrow/feather.py:156: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = Table.from_pandas(df, preserve_index=preserve_index)
